# Homework: Agentic RAG
## Setup

In [1]:
from dotenv import load_dotenv
from openai import OpenAI
from rag import RAGBase
from ingest import load_lessons_data, build_index
from gitsource import chunk_documents

In [2]:
load_dotenv()
openai_client = OpenAI()

## Indexing and searching

In [3]:
documents = load_lessons_data()
index = build_index(documents)

In [4]:
len(documents)

72

In [5]:
question = "How does the agentic loop keep calling the model until it stops?"

In [6]:
search_results = index.search(question, num_results=5)

In [7]:
search_results[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

## RAG

In [8]:
assistant = RAGBase(index, openai_client)

In [9]:
response = assistant.rag(question)

In [10]:
response.usage.prompt_tokens

7142

## Chunking

In [11]:
chunks = chunk_documents(documents, size=2000, step=1000)

In [12]:
len(chunks)

295

In [13]:
chunked_index = build_index(chunks)

In [14]:
chunked_assistant = RAGBase(chunked_index, openai_client)

In [15]:
response = chunked_assistant.rag(question)

In [16]:
response.usage.prompt_tokens

2325

## Agent

In [17]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [18]:
def search(query: str) -> dict[str, str]:
	"""
	Search the lesson database for entries matching the given query.
	"""
	return chunked_index.search(
		query,
		num_results=5,
		boost_dict={'filename': 3.0, 'content': 0.5}
	)

In [19]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [20]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [21]:
instructions = '''
You're a course teaching assistant. Answer the student's question using the search tool.
Make multiple searches with different keywords before answering.
'''

In [22]:
runner = OpenAIResponsesRunner(
	tools=agent_tools,
	developer_prompt=instructions,
	chat_interface=chat_interface,
	llm_client=OpenAIClient(model='gpt-4o-mini')
)

In [24]:
result = runner.loop(
	prompt="How does the agentic loop work, and how is it different from plain RAG?",
	callback=callback,
)